In [ ]:
import os
print("=== Files under /kaggle/input ===")
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        p = os.path.join(root, f)
        print(f"{p}   ({os.path.getsize(p)/1e6:.1f} MB)")

In [ ]:
import pandas as pd
BASE = '/kaggle/input/datasets/codewithtuqi/urdu-twitter-datasets'
proc = pd.read_excel(f'{BASE}/cleaned_urdu_dataset_1M_processed.xlsx', engine='openpyxl')
proc.to_parquet('/kaggle/working/processed.parquet', index=False)
print("rows:", len(proc))

In [ ]:
import pandas as pd, numpy as np, ast, re, unicodedata, pickle
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
K=20
proc=pd.read_parquet('/kaggle/working/processed.parquet')
VS={'\uFE0F','\uFE0E'}; SKIN={chr(c) for c in range(0x1F3FB,0x1F400)}
def norm_emoji(e): return ''.join(c for c in e if c not in VS and c not in SKIN)
def parse_list(x):
    if isinstance(x,list): return x
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    try:
        v=ast.literal_eval(x); return v if isinstance(v,list) else [v]
    except Exception: return []
uniq=proc['Unique_Emojis'].apply(parse_list).apply(lambda l:list(dict.fromkeys(ne for e in l if (ne:=norm_emoji(e)))))
cnt=Counter(e for row in uniq for e in row)
VOCAB=[e for e,_ in cnt.most_common(K)]; vset=set(VOCAB)
URL=re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    if not isinstance(t,str): return ''
    t=unicodedata.normalize('NFC',t); t=URL.sub(' ',t)
    t=''.join(ch for ch in t if not (0xE000<=ord(ch)<=0xF8FF)); return re.sub(r'\s+',' ',t).strip()
df=pd.DataFrame({'text':proc['Urdu_Text'].apply(clean),'emojis':uniq.apply(lambda l:[e for e in l if e in vset])})
df=df[(df['emojis'].apply(len)>0)&(df['text'].str.len()>=2)].reset_index(drop=True)
Y=MultiLabelBinarizer(classes=VOCAB).fit_transform(df['emojis']).astype('int8')
idx=np.arange(len(df)); tr,tmp=train_test_split(idx,test_size=0.2,random_state=42)
va,te=train_test_split(tmp,test_size=0.5,random_state=42)
split=np.array(['train']*len(df),dtype=object); split[va]='val'; split[te]='test'
pd.DataFrame({'text':df['text'],'labels':list(Y),'split':split}).to_parquet('/kaggle/working/train_ready_k20.parquet',index=False)
pickle.dump(VOCAB,open('/kaggle/working/vocab_k20.pkl','wb'))
print("k20 ready:", len(df), "rows")

In [ ]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
import numpy as np, pandas as pd, pickle, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score
VOCAB=pickle.load(open('/kaggle/working/vocab_k20.pkl','rb')); NUM=len(VOCAB)
df=pd.read_parquet('/kaggle/working/train_ready_k20.parquet')
MODEL,MAXLEN="xlm-roberta-base",64
tok=AutoTokenizer.from_pretrained(MODEL)
def to_ds(s):
    sub=df[df.split==s]
    return Dataset.from_dict({'text':sub['text'].tolist(),'labels':[np.asarray(x,dtype=np.float32) for x in sub['labels']]})
ds={s:to_ds(s) for s in ['train','val','test']}
ds={s:d.map(lambda b:tok(b['text'],truncation=True,max_length=MAXLEN),batched=True,remove_columns=['text']) for s,d in ds.items()}
class ASL(torch.nn.Module):
    def __init__(s,gn=4,gp=1,clip=0.05,eps=1e-8): super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logits,y):
        xp=torch.sigmoid(logits); xn=1-xp
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        loss=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y); loss*=(1-pt).pow(g)
        return -loss.sum(1).mean()
asl=ASL()
class MLTrainer(Trainer):
    def compute_loss(s,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs); loss=asl(out.logits,labels)
        return (loss,out) if return_outputs else loss
def compute_metrics(ep):
    logits,labels=ep; probs=1/(1+np.exp(-logits)); preds=(probs>=0.5).astype(int)
    rows=np.arange(len(labels))[:,None]; top3=np.argsort(-probs,1)[:,:3]
    return {'micro_f1':f1_score(labels,preds,average='micro',zero_division=0),
            'macro_f1':f1_score(labels,preds,average='macro',zero_division=0),
            'subset_acc':accuracy_score(labels,preds),
            'P@1':float(labels[rows,probs.argmax(1)[:,None]].mean()),
            'acc@3':float(labels[rows,top3].any(1).mean())}
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=NUM,problem_type="multi_label_classification")
args=TrainingArguments(output_dir='/kaggle/working/xlmr_k20',num_train_epochs=2,learning_rate=2e-5,fp16=True,
    per_device_train_batch_size=128,per_device_eval_batch_size=256,eval_strategy='epoch',save_strategy='epoch',
    load_best_model_at_end=True,metric_for_best_model='acc@3',logging_steps=200,report_to='none',dataloader_num_workers=2)
trainer_k20=MLTrainer(model=model,args=args,train_dataset=ds['train'],eval_dataset=ds['val'],
    processing_class=tok,compute_metrics=compute_metrics,data_collator=DataCollatorWithPadding(tok))
trainer_k20.train()
print("\n=== K=20 TEST ==="); print(trainer_k20.evaluate(ds['test']))
trainer_k20.save_model('/kaggle/working/xlmr_k20_best')

In [ ]:
from openpyxl import load_workbook

def peek_xlsx(path, n=8):
    print(f"\n{'='*70}\nPEEK: {path}\n{'='*70}")
    wb = load_workbook(path, read_only=True, data_only=True)
    print("Sheets:", wb.sheetnames)
    ws = wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    header = next(rows)
    print("Columns:", header)
    print(f"\nFirst {n} data rows (raw repr — shows exact emoji encoding):")
    for i, r in enumerate(rows):
        if i >= n:
            break
        for col, val in zip(header, r):
            print(f"   [{col}] {repr(val)}")
        print("   " + "-" * 45)
    wb.close()

In [ ]:
BASE = '/kaggle/input/datasets/codewithtuqi/urdu-twitter-datasets'

peek_xlsx(f'{BASE}/Urdu Tweets Dataset.xlsx')
peek_xlsx(f'{BASE}/cleaned_urdu_dataset_1M_processed.xlsx')

In [ ]:
import pandas as pd, ast, re
from collections import Counter

BASE = '/kaggle/input/datasets/codewithtuqi/urdu-twitter-datasets'

# --- load once (big xlsx: give it a few minutes) and cache as parquet ---
proc = pd.read_excel(f'{BASE}/cleaned_urdu_dataset_1M_processed.xlsx', engine='openpyxl')
proc.to_parquet('/kaggle/working/processed.parquet', index=False)
print(f"TOTAL ROWS: {len(proc):,}")          # truncation check: ~1,048,575 == Excel truncated
print("Columns:", list(proc.columns))

# --- safe parser for the stringified lists ---
def parse_list(x):
    if isinstance(x, list): return x
    if x is None or (isinstance(x, float) and pd.isna(x)): return []
    try:
        v = ast.literal_eval(x)
        return v if isinstance(v, list) else [v]
    except Exception:
        return []

uniq = proc['Unique_Emojis'].apply(parse_list)
desc = proc['Emoji_Description'].apply(parse_list)

# --- 1. label cardinality (emojis per tweet) from Unique_Emojis ---
card = uniq.apply(len)
print("\n--- emojis per tweet (Unique_Emojis) ---")
print(card.value_counts().sort_index().head(12))
print("rows with ZERO emojis:", int((card == 0).sum()))

# --- 2. how often the two label columns disagree in COUNT ---
diff = (uniq.apply(len) - desc.apply(len))
print("\n--- (len Unique_Emojis) - (len Emoji_Description) ---")
print(diff.value_counts().sort_index().head(15))
print("rows where counts differ:", int((diff != 0).sum()),
      f"({(diff != 0).mean()*100:.1f}%)")

# --- 3. vocabulary + frequency from Unique_Emojis ---
all_e = Counter(e for row in uniq for e in row)
print("\nUNIQUE emoji characters in corpus:", len(all_e))
print("Top 30:", all_e.most_common(30))

# --- 4. empty-text checks ---
def blank(s): return (s is None) or (str(s).strip() == '')
print("\nUrdu_Text blank:", int(proc['Urdu_Text'].apply(blank).sum()))
print("Urdu_Text_Cleaned blank:", int(proc['Urdu_Text_Cleaned'].apply(blank).sum()))

In [ ]:
import pandas as pd, ast, numpy as np
from collections import Counter

proc = pd.read_parquet('/kaggle/working/processed.parquet')

def parse_list(x):
    if isinstance(x, list): return x
    if x is None or (isinstance(x, float) and pd.isna(x)): return []
    try:
        v = ast.literal_eval(x); return v if isinstance(v, list) else [v]
    except Exception:
        return []

# --- emoji normalization: drop variation selectors + skin-tone modifiers ---
VS   = {'\uFE0F', '\uFE0E'}
SKIN = {chr(c) for c in range(0x1F3FB, 0x1F400)}   # 5 skin tones
def norm_emoji(e):
    return ''.join(ch for ch in e if ch not in VS and ch not in SKIN)

uniq      = proc['Unique_Emojis'].apply(parse_list)
uniq_norm = uniq.apply(lambda lst: list(dict.fromkeys(
                 ne for e in lst if (ne := norm_emoji(e)))))

cnt = Counter(e for row in uniq_norm for e in row)
total_occ = sum(cnt.values())
print("unique emojis  BEFORE norm:", len({e for row in uniq for e in row}))
print("unique emojis  AFTER  norm:", len(cnt))
print("total emoji occurrences   :", f"{total_occ:,}")

# --- coverage curve: how many emojis to cover X% of all occurrences ---
freqs = np.array(sorted(cnt.values(), reverse=True))
cum   = np.cumsum(freqs) / total_occ
for t in (0.80, 0.90, 0.95, 0.99):
    k = int(np.searchsorted(cum, t) + 1)
    print(f"  top-{k:<4} covers {t*100:.0f}%")

print("\nTop 30 after norm:", cnt.most_common(30))

# --- for candidate K: how many rows stay usable ---
for K in (50, 100, 200, 300):
    topK = {e for e, _ in cnt.most_common(K)}
    full = uniq_norm.apply(lambda l: all(e in topK for e in l)).mean()
    anyc = uniq_norm.apply(lambda l: any(e in topK for e in l)).mean()
    print(f"K={K:<4} rows fully in-vocab {full*100:5.1f}% | rows with >=1 in-vocab {anyc*100:5.1f}%")

import pickle
pickle.dump(uniq_norm.tolist(), open('/kaggle/working/labels_norm.pkl','wb'))

In [ ]:
import pandas as pd, numpy as np, ast, re, unicodedata, pickle
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

K = 100   # <-- the only knob; rerun with 50 / 200 later for the scaling table

proc = pd.read_parquet('/kaggle/working/processed.parquet')

VS={'\uFE0F','\uFE0E'}; SKIN={chr(c) for c in range(0x1F3FB,0x1F400)}
def norm_emoji(e): return ''.join(c for c in e if c not in VS and c not in SKIN)
def parse_list(x):
    if isinstance(x,list): return x
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    try:
        v=ast.literal_eval(x); return v if isinstance(v,list) else [v]
    except Exception: return []

uniq = proc['Unique_Emojis'].apply(parse_list).apply(
        lambda l: list(dict.fromkeys(ne for e in l if (ne:=norm_emoji(e)))))
cnt   = Counter(e for row in uniq for e in row)
VOCAB = [e for e,_ in cnt.most_common(K)]; vset = set(VOCAB)

# --- light cleaning only ---
URL = re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    if not isinstance(t,str): return ''
    t = unicodedata.normalize('NFC', t)
    t = URL.sub(' ', t)
    t = ''.join(ch for ch in t if not (0xE000 <= ord(ch) <= 0xF8FF))  # private-use junk
    return re.sub(r'\s+',' ', t).strip()

df = pd.DataFrame({'text': proc['Urdu_Text'].apply(clean),
                   'emojis': uniq.apply(lambda l: [e for e in l if e in vset])})
before = len(df)
df = df[(df['emojis'].apply(len) > 0) & (df['text'].str.len() >= 2)].reset_index(drop=True)
print(f"rows kept: {len(df):,} / {before:,} ({len(df)/before*100:.1f}%)")

mlb = MultiLabelBinarizer(classes=VOCAB)
Y = mlb.fit_transform(df['emojis']).astype('int8')
print("label matrix:", Y.shape, "| avg labels/row:", round(float(Y.sum(1).mean()),3))
colsum = Y.sum(0)
print("class freq  -> max:", int(colsum.max()), " median:", int(np.median(colsum)),
      " min:", int(colsum.min()))

idx = np.arange(len(df))
tr, tmp = train_test_split(idx, test_size=0.2, random_state=42)
va, te  = train_test_split(tmp, test_size=0.5, random_state=42)
split = np.array(['train']*len(df), dtype=object)
split[va] = 'val'; split[te] = 'test'
print(f"train {len(tr):,} | val {len(va):,} | test {len(te):,}")

out = pd.DataFrame({'text': df['text'], 'labels': list(Y), 'split': split})
out.to_parquet('/kaggle/working/train_ready.parquet', index=False)
pickle.dump(VOCAB, open('/kaggle/working/vocab.pkl','wb'))
print("saved -> train_ready.parquet, vocab.pkl\n")
print(out[['text','split']].head(3).to_string())

In [ ]:
import numpy as np, pandas as pd, pickle, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score

VOCAB = pickle.load(open('/kaggle/working/vocab.pkl','rb'))
NUM_LABELS = len(VOCAB)
df = pd.read_parquet('/kaggle/working/train_ready.parquet')
# df = df.groupby('split', group_keys=False).sample(frac=0.05, random_state=0)  # <- uncomment for a 5-min smoke test

MODEL, MAXLEN = "xlm-roberta-base", 64
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(split):
    sub = df[df.split == split]
    return Dataset.from_dict({
        'text':   sub['text'].tolist(),
        'labels': [np.asarray(x, dtype=np.float32) for x in sub['labels']]})

ds = {s: to_ds(s) for s in ['train','val','test']}
ds = {s: d.map(lambda b: tok(b['text'], truncation=True, max_length=MAXLEN),
               batched=True, remove_columns=['text']) for s, d in ds.items()}

# ---- Asymmetric Loss (Ben-Baruch et al., 2021): down-weights easy negatives ----
class ASL(torch.nn.Module):
    def __init__(self, g_neg=4, g_pos=1, clip=0.05, eps=1e-8):
        super().__init__(); self.gn,self.gp,self.clip,self.eps = g_neg,g_pos,clip,eps
    def forward(self, logits, y):
        xp = torch.sigmoid(logits); xn = 1 - xp
        if self.clip > 0: xn = (xn + self.clip).clamp(max=1)
        loss = y*torch.log(xp.clamp(min=self.eps)) + (1-y)*torch.log(xn.clamp(min=self.eps))
        pt   = xp*y + xn*(1-y)
        gamma = self.gp*y + self.gn*(1-y)
        loss *= (1-pt).pow(gamma)
        return -loss.sum(1).mean()
asl = ASL()

class MLTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = asl(out.logits, labels)
        return (loss, out) if return_outputs else loss

def compute_metrics(ep):
    logits, labels = ep
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    rows  = np.arange(len(labels))[:,None]
    top3  = np.argsort(-probs,1)[:,:3]
    return {
        'micro_f1':   f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1':   f1_score(labels, preds, average='macro', zero_division=0),
        'subset_acc': accuracy_score(labels, preds),
        'P@1':  float(labels[rows, probs.argmax(1)[:,None]].mean()),
        'acc@3': float(labels[rows, top3].any(1).mean()),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=NUM_LABELS, problem_type="multi_label_classification")

args = TrainingArguments(
    output_dir='/kaggle/working/xlmr_emoji',
    num_train_epochs=2, learning_rate=2e-5, fp16=True,
    per_device_train_batch_size=128, per_device_eval_batch_size=256,
    eval_strategy='epoch', save_strategy='epoch',      # if this errors: evaluation_strategy
    load_best_model_at_end=True, metric_for_best_model='micro_f1',
    logging_steps=200, report_to='none', dataloader_num_workers=2)

trainer = MLTrainer(model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['val'],
    processing_class=tok, compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tok))

trainer.train()
print("\n=== TEST ==="); print(trainer.evaluate(ds['test']))
trainer.save_model('/kaggle/working/xlmr_emoji_best')

In [ ]:
import torch, torch.nn as nn, numpy as np, pickle
from transformers import AutoModel, AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding

MODEL = "xlm-roberta-base"
VOCAB = pickle.load(open('/kaggle/working/vocab.pkl','rb'))
try:
    import emoji as emojilib
except Exception:
    import os; os.system('pip -q install emoji'); import emoji as emojilib

# emoji char -> readable description ("😂" -> "face with tears of joy")
def name_of(ch):
    d = emojilib.demojize(ch)
    return d.strip(':').replace('_',' ') if d.startswith(':') else ch
descs = [name_of(e) for e in VOCAB]

# --- semantic init for the emoji tower: encode descriptions with the same encoder ---
enc_init = AutoModel.from_pretrained(MODEL).eval()
def meanpool(o, m): 
    m = m.unsqueeze(-1).float(); return (o.last_hidden_state*m).sum(1)/m.sum(1).clamp(min=1e-9)
with torch.no_grad():
    chunks=[]
    for i in range(0,len(descs),128):
        b = tok(descs[i:i+128], padding=True, truncation=True, max_length=16, return_tensors='pt')
        chunks.append(meanpool(enc_init(**b), b['attention_mask']))
    emoji_init = torch.cat(chunks).float()      # [K, hidden]
del enc_init

class TwoTower(nn.Module):
    def __init__(self, model_name, emoji_init, temp=0.05):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.emoji = nn.Parameter(emoji_init.clone())
        self.logit_scale = nn.Parameter(torch.tensor(1/temp).log())   # CLIP-style learnable temp
        self.loss_fn = ASL()
    def forward(self, input_ids, attention_mask, labels=None):
        t = nn.functional.normalize(meanpool(self.encoder(input_ids=input_ids,
                                    attention_mask=attention_mask), attention_mask), dim=-1)
        e = nn.functional.normalize(self.emoji, dim=-1)
        logits = self.logit_scale.exp().clamp(max=100) * (t @ e.t())   # [B, K]
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits}

model = TwoTower(MODEL, emoji_init)

def compute_metrics(ep):
    from sklearn.metrics import f1_score, accuracy_score
    logits, labels = ep; probs = 1/(1+np.exp(-logits)); preds=(probs>=0.5).astype(int)
    rows=np.arange(len(labels))[:,None]; top3=np.argsort(-probs,1)[:,:3]
    return {'micro_f1':f1_score(labels,preds,average='micro',zero_division=0),
            'macro_f1':f1_score(labels,preds,average='macro',zero_division=0),
            'subset_acc':accuracy_score(labels,preds),
            'P@1':float(labels[rows,probs.argmax(1)[:,None]].mean()),
            'acc@3':float(labels[rows,top3].any(1).mean())}

args = TrainingArguments(output_dir='/kaggle/working/twotower',
    num_train_epochs=2, learning_rate=2e-5, fp16=True,
    per_device_train_batch_size=128, per_device_eval_batch_size=256,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='acc@3', logging_steps=200, report_to='none', dataloader_num_workers=2)

trainer2 = Trainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'],
    processing_class=tok, compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tok))
trainer2.train()
print("\n=== TWO-TOWER TEST ==="); print(trainer2.evaluate(ds['test']))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
pv = trainer.predict(ds['val']); pt = trainer.predict(ds['test'])
probs_v=1/(1+np.exp(-pv.predictions)); yval=pv.label_ids.astype(int)
probs_t=1/(1+np.exp(-pt.predictions)); ytest=pt.label_ids.astype(int)
grid=np.arange(0.05,0.60,0.02)
thr=np.array([max(grid,key=lambda t:f1_score(yval[:,c],(probs_v[:,c]>=t).astype(int),zero_division=0))
              for c in range(probs_v.shape[1])])
predt=(probs_t>=thr).astype(int)
print("tuned micro-F1:", round(f1_score(ytest,predt,average='micro',zero_division=0),4))
print("tuned macro-F1:", round(f1_score(ytest,predt,average='macro',zero_division=0),4))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
grid = np.arange(0.05, 0.60, 0.02)
gt = max(grid, key=lambda t: f1_score(yval, (probs_v>=t).astype(int), average='micro', zero_division=0))
pg = (probs_t >= gt).astype(int)
print("global thr", round(float(gt),2),
      "| micro-F1", round(f1_score(ytest, pg, average='micro', zero_division=0),4),
      "| macro-F1", round(f1_score(ytest, pg, average='macro', zero_division=0),4))

In [ ]:
import pickle, re, unicodedata, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "/kaggle/working/xlmr_emoji_best"     # <- switch to e5 dir later if you retrain
VOCAB = pickle.load(open("/kaggle/working/vocab.pkl","rb"))
try:
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
except Exception:
    tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
dev = "cuda" if torch.cuda.is_available() else "cpu"; model.to(dev)

URL = re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    t = unicodedata.normalize('NFC', str(t)); t = URL.sub(' ', t)
    t = ''.join(c for c in t if not (0xE000 <= ord(c) <= 0xF8FF))
    return re.sub(r'\s+',' ',t).strip()

@torch.no_grad()
def predict(text, k=5):
    enc = tok(clean(text), truncation=True, max_length=64, return_tensors='pt').to(dev)
    probs = torch.sigmoid(model(**enc).logits[0]).cpu().numpy()
    idx = np.argsort(-probs)[:k]
    return [(VOCAB[i], round(float(probs[i]),3)) for i in idx]

# quick sanity set — type known-sentiment Urdu and see if the top emojis make sense
tests = [
    "آج بہت خوشی کا دن ہے",          # happy
    "میں بہت اداس اور پریشان ہوں",    # sad
    "یہ بات بہت مزاحیہ ہے ہا ہا",     # funny
    "بہت بہت مبارک ہو دوست",          # congrats
    "اللہ آپ کو ہمیشہ خوش رکھے",      # blessing/love
    "میں تم سے بہت محبت کرتا ہوں",    # love
]
for t in tests:
    print(t)
    print("   ", predict(t, k=5), "\n")

In [ ]:
%%writefile /kaggle/working/predict.py
import sys, pickle, re, unicodedata, argparse, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL_DIR = "/kaggle/working/xlmr_emoji_best"
VOCAB = pickle.load(open("/kaggle/working/vocab.pkl","rb"))
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
URL = re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    t = unicodedata.normalize('NFC', str(t)); t = URL.sub(' ', t)
    t = ''.join(c for c in t if not (0xE000<=ord(c)<=0xF8FF)); return re.sub(r'\s+',' ',t).strip()
@torch.no_grad()
def predict(text, k):
    enc = tok(clean(text), truncation=True, max_length=64, return_tensors='pt')
    p = torch.sigmoid(model(**enc).logits[0]).numpy()
    return [(VOCAB[i], float(p[i])) for i in np.argsort(-p)[:k]]
if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("text", nargs="*"); ap.add_argument("-k", type=int, default=5)
    a = ap.parse_args()
    if a.text:
        for e,p in predict(" ".join(a.text), a.k): print(f"  {e}   {p:.3f}")
    else:
        print("Type an Urdu sentence (blank to quit):")
        while (t := input("\n> ").strip()):
            for e,p in predict(t, a.k): print(f"  {e}   {p:.3f}")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
scale = trainer2.model.logit_scale.exp().item()
pt = trainer2.predict(ds['test'])
cos = pt.predictions / scale; y = pt.label_ids.astype(int)
rows = np.arange(len(y))[:,None]
print("P@1  :", round(float(y[rows, cos.argmax(1)[:,None]].mean()),4))
print("acc@3:", round(float(y[rows, np.argsort(-cos,1)[:,:3]].any(1).mean()),4))
print(f"\n{'thr':>5}{'pred>=1%':>10}{'hit%':>7}{'micro-P':>9}{'micro-R':>9}{'micro-F1':>10}")
for thr in [0.5,0.7,0.9]:
    pred=(cos>=thr).astype(int)
    has=(pred.sum(1)>0).mean()*100; hit=((pred&y).sum(1)>0).mean()*100
    P=precision_score(y,pred,average='micro',zero_division=0)
    R=recall_score(y,pred,average='micro',zero_division=0)
    F=f1_score(y,pred,average='micro',zero_division=0)
    print(f"{thr:>5}{has:>10.1f}{hit:>7.1f}{P:>9.3f}{R:>9.3f}{F:>10.3f}")

In [ ]:
import pandas as pd, numpy as np, ast, re, unicodedata, pickle
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

K = 20
proc = pd.read_parquet('/kaggle/working/processed.parquet')
VS={'\uFE0F','\uFE0E'}; SKIN={chr(c) for c in range(0x1F3FB,0x1F400)}
def norm_emoji(e): return ''.join(c for c in e if c not in VS and c not in SKIN)
def parse_list(x):
    if isinstance(x,list): return x
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    try:
        v=ast.literal_eval(x); return v if isinstance(v,list) else [v]
    except Exception: return []
uniq = proc['Unique_Emojis'].apply(parse_list).apply(
        lambda l: list(dict.fromkeys(ne for e in l if (ne:=norm_emoji(e)))))
cnt=Counter(e for row in uniq for e in row)
VOCAB=[e for e,_ in cnt.most_common(K)]; vset=set(VOCAB)
URL=re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    if not isinstance(t,str): return ''
    t=unicodedata.normalize('NFC',t); t=URL.sub(' ',t)
    t=''.join(ch for ch in t if not (0xE000<=ord(ch)<=0xF8FF)); return re.sub(r'\s+',' ',t).strip()
df=pd.DataFrame({'text':proc['Urdu_Text'].apply(clean),
                 'emojis':uniq.apply(lambda l:[e for e in l if e in vset])})
before=len(df)
df=df[(df['emojis'].apply(len)>0)&(df['text'].str.len()>=2)].reset_index(drop=True)
print(f"rows kept: {len(df):,}/{before:,} ({len(df)/before*100:.1f}%)")
Y=MultiLabelBinarizer(classes=VOCAB).fit_transform(df['emojis']).astype('int8')
print("labels:",Y.shape,"| avg/row:",round(float(Y.sum(1).mean()),3),"| class freq min:",int(Y.sum(0).min()))
idx=np.arange(len(df)); tr,tmp=train_test_split(idx,test_size=0.2,random_state=42)
va,te=train_test_split(tmp,test_size=0.5,random_state=42)
split=np.array(['train']*len(df),dtype=object); split[va]='val'; split[te]='test'
pd.DataFrame({'text':df['text'],'labels':list(Y),'split':split}).to_parquet('/kaggle/working/train_ready_k20.parquet',index=False)
pickle.dump(VOCAB,open('/kaggle/working/vocab_k20.pkl','wb'))
print("saved k20 files | train/val/test:",len(tr),len(va),len(te))

In [ ]:
import gc, torch
for v in ['trainer','trainer2','trainer5','trainer_k20','model','model5','enc_init','pv','pt']:
    if v in globals():
        try: del globals()[v]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
print(round(torch.cuda.memory_allocated()/1e9,2), "GB still allocated")

In [ ]:

import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
mport numpy as np, pandas as pd, pickle, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score

VOCAB=pickle.load(open('/kaggle/working/vocab_k20.pkl','rb')); NUM=len(VOCAB)
df=pd.read_parquet('/kaggle/working/train_ready_k20.parquet')
MODEL,MAXLEN="xlm-roberta-base",64
tok=AutoTokenizer.from_pretrained(MODEL)
def to_ds(s):
    sub=df[df.split==s]
    return Dataset.from_dict({'text':sub['text'].tolist(),
                              'labels':[np.asarray(x,dtype=np.float32) for x in sub['labels']]})
ds={s:to_ds(s) for s in ['train','val','test']}
ds={s:d.map(lambda b:tok(b['text'],truncation=True,max_length=MAXLEN),
            batched=True,remove_columns=['text']) for s,d in ds.items()}

class ASL(torch.nn.Module):
    def __init__(s,gn=4,gp=1,clip=0.05,eps=1e-8): super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logits,y):
        xp=torch.sigmoid(logits); xn=1-xp
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        loss=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y); loss*=(1-pt).pow(g)
        return -loss.sum(1).mean()
asl=ASL()
class MLTrainer(Trainer):
    def compute_loss(s,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs)
        loss=asl(out.logits,labels); return (loss,out) if return_outputs else loss
def compute_metrics(ep):
    logits,labels=ep; probs=1/(1+np.exp(-logits)); preds=(probs>=0.5).astype(int)
    rows=np.arange(len(labels))[:,None]; top3=np.argsort(-probs,1)[:,:3]
    return {'micro_f1':f1_score(labels,preds,average='micro',zero_division=0),
            'macro_f1':f1_score(labels,preds,average='macro',zero_division=0),
            'subset_acc':accuracy_score(labels,preds),
            'P@1':float(labels[rows,probs.argmax(1)[:,None]].mean()),
            'acc@3':float(labels[rows,top3].any(1).mean())}
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=NUM,problem_type="multi_label_classification")
args=TrainingArguments(output_dir='/kaggle/working/xlmr_k20',num_train_epochs=2,learning_rate=2e-5,fp16=True,
    per_device_train_batch_size=128,per_device_eval_batch_size=256,eval_strategy='epoch',save_strategy='epoch',
    load_best_model_at_end=True,metric_for_best_model='acc@3',logging_steps=200,report_to='none',dataloader_num_workers=2)
trainer_k20=MLTrainer(model=model,args=args,train_dataset=ds['train'],eval_dataset=ds['val'],
    processing_class=tok,compute_metrics=compute_metrics,data_collator=DataCollatorWithPadding(tok))
trainer_k20.train()
print("\n=== K=20 TEST (compare acc@3 to your thesis 72%) ===")
print(trainer_k20.evaluate(ds['test'])); trainer_k20.save_model('/kaggle/working/xlmr_k20_best')

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd, numpy as np, ast, re, unicodedata, pickle, torch
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, accuracy_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

BASE = '/kaggle/input/datasets/codewithtuqi/urdu-twitter-datasets'
K = 20

# ===== STEP 1: Excel -> parquet =====
proc = pd.read_excel(f'{BASE}/cleaned_urdu_dataset_1M_processed.xlsx', engine='openpyxl')
proc.to_parquet('/kaggle/working/processed.parquet', index=False)
print("STEP 1 done — rows:", len(proc))

# ===== STEP 2: build K=20 data =====
VS={'\uFE0F','\uFE0E'}; SKIN={chr(c) for c in range(0x1F3FB,0x1F400)}
def norm_emoji(e): return ''.join(c for c in e if c not in VS and c not in SKIN)
def parse_list(x):
    if isinstance(x,list): return x
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    try:
        v=ast.literal_eval(x); return v if isinstance(v,list) else [v]
    except Exception: return []
uniq=proc['Unique_Emojis'].apply(parse_list).apply(lambda l:list(dict.fromkeys(ne for e in l if (ne:=norm_emoji(e)))))
cnt=Counter(e for row in uniq for e in row)
VOCAB=[e for e,_ in cnt.most_common(K)]; vset=set(VOCAB)
pickle.dump(VOCAB, open('/kaggle/working/vocab_k20.pkl','wb'))
URL=re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    if not isinstance(t,str): return ''
    t=unicodedata.normalize('NFC',t); t=URL.sub(' ',t)
    t=''.join(ch for ch in t if not (0xE000<=ord(ch)<=0xF8FF)); return re.sub(r'\s+',' ',t).strip()
df=pd.DataFrame({'text':proc['Urdu_Text'].apply(clean),'emojis':uniq.apply(lambda l:[e for e in l if e in vset])})
df=df[(df['emojis'].apply(len)>0)&(df['text'].str.len()>=2)].reset_index(drop=True)
Y=MultiLabelBinarizer(classes=VOCAB).fit_transform(df['emojis']).astype('int8')
idx=np.arange(len(df)); tr,tmp=train_test_split(idx,test_size=0.2,random_state=42)
va,te=train_test_split(tmp,test_size=0.5,random_state=42)
split=np.array(['train']*len(df),dtype=object); split[va]='val'; split[te]='test'
data=pd.DataFrame({'text':df['text'],'labels':list(Y),'split':split})
print("STEP 2 done — k20 rows:", len(df), "| classes:", len(VOCAB))

# ===== STEP 3: train K=20 =====
NUM=len(VOCAB); MODEL,MAXLEN="xlm-roberta-base",64
tok=AutoTokenizer.from_pretrained(MODEL)
def to_ds(s):
    sub=data[data.split==s]
    return Dataset.from_dict({'text':sub['text'].tolist(),
                              'labels':[np.asarray(x,dtype=np.float32) for x in sub['labels']]})
ds={s:to_ds(s) for s in ['train','val','test']}
ds={s:d.map(lambda b:tok(b['text'],truncation=True,max_length=MAXLEN),
            batched=True,remove_columns=['text']) for s,d in ds.items()}
class ASL(torch.nn.Module):
    def __init__(s,gn=4,gp=1,clip=0.05,eps=1e-8): super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logits,y):
        xp=torch.sigmoid(logits); xn=1-xp
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        loss=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y); loss*=(1-pt).pow(g)
        return -loss.sum(1).mean()
asl=ASL()
class MLTrainer(Trainer):
    def compute_loss(s,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs); loss=asl(out.logits,labels)
        return (loss,out) if return_outputs else loss
def compute_metrics(ep):
    logits,labels=ep; probs=1/(1+np.exp(-logits)); preds=(probs>=0.5).astype(int)
    rows=np.arange(len(labels))[:,None]; top3=np.argsort(-probs,1)[:,:3]
    return {'micro_f1':f1_score(labels,preds,average='micro',zero_division=0),
            'macro_f1':f1_score(labels,preds,average='macro',zero_division=0),
            'subset_acc':accuracy_score(labels,preds),
            'P@1':float(labels[rows,probs.argmax(1)[:,None]].mean()),
            'acc@3':float(labels[rows,top3].any(1).mean())}
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=NUM,
        problem_type="multi_label_classification")
args=TrainingArguments(output_dir='/kaggle/working/xlmr_k20',num_train_epochs=2,learning_rate=2e-5,fp16=True,
    per_device_train_batch_size=128,per_device_eval_batch_size=256,eval_strategy='epoch',save_strategy='epoch',
    load_best_model_at_end=True,metric_for_best_model='acc@3',logging_steps=200,report_to='none',dataloader_num_workers=2)
trainer_k20=MLTrainer(model=model,args=args,train_dataset=ds['train'],eval_dataset=ds['val'],
    processing_class=tok,compute_metrics=compute_metrics,data_collator=DataCollatorWithPadding(tok))
trainer_k20.train()
print("\n=== K=20 TEST ==="); print(trainer_k20.evaluate(ds['test']))
trainer_k20.save_model('/kaggle/working/xlmr_k20_best')
print("ALL DONE — saved to /kaggle/working/xlmr_k20_best")

In [ ]:
pickle.dump(VOCAB, open(f'/kaggle/working/vocab_k{K}.pkl','wb'))
# ...
args=TrainingArguments(output_dir=f'/kaggle/working/xlmr_k{K}', ...   # just the output_dir part
# ...
trainer_k20.save_model(f'/kaggle/working/xlmr_k{K}_best')

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd, numpy as np, ast, re, unicodedata, pickle, torch
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, accuracy_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

BASE = '/kaggle/input/datasets/codewithtuqi/urdu-twitter-datasets'
K = 50          # <<< CHANGE ONLY THIS NUMBER (50, then 200). Nothing else.

# STEP 1: Excel -> parquet
proc = pd.read_excel(f'{BASE}/cleaned_urdu_dataset_1M_processed.xlsx', engine='openpyxl')
proc.to_parquet('/kaggle/working/processed.parquet', index=False)
print("STEP 1 done — rows:", len(proc))

# STEP 2: build data for this K
VS={'\uFE0F','\uFE0E'}; SKIN={chr(c) for c in range(0x1F3FB,0x1F400)}
def norm_emoji(e): return ''.join(c for c in e if c not in VS and c not in SKIN)
def parse_list(x):
    if isinstance(x,list): return x
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    try:
        v=ast.literal_eval(x); return v if isinstance(v,list) else [v]
    except Exception: return []
uniq=proc['Unique_Emojis'].apply(parse_list).apply(lambda l:list(dict.fromkeys(ne for e in l if (ne:=norm_emoji(e)))))
cnt=Counter(e for row in uniq for e in row)
VOCAB=[e for e,_ in cnt.most_common(K)]; vset=set(VOCAB)
pickle.dump(VOCAB, open(f'/kaggle/working/vocab_k{K}.pkl','wb'))
URL=re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    if not isinstance(t,str): return ''
    t=unicodedata.normalize('NFC',t); t=URL.sub(' ',t)
    t=''.join(ch for ch in t if not (0xE000<=ord(ch)<=0xF8FF)); return re.sub(r'\s+',' ',t).strip()
df=pd.DataFrame({'text':proc['Urdu_Text'].apply(clean),'emojis':uniq.apply(lambda l:[e for e in l if e in vset])})
df=df[(df['emojis'].apply(len)>0)&(df['text'].str.len()>=2)].reset_index(drop=True)
Y=MultiLabelBinarizer(classes=VOCAB).fit_transform(df['emojis']).astype('int8')
idx=np.arange(len(df)); tr,tmp=train_test_split(idx,test_size=0.2,random_state=42)
va,te=train_test_split(tmp,test_size=0.5,random_state=42)
split=np.array(['train']*len(df),dtype=object); split[va]='val'; split[te]='test'
data=pd.DataFrame({'text':df['text'],'labels':list(Y),'split':split})
print(f"STEP 2 done — K={K} rows:", len(df))

# STEP 3: train
NUM=len(VOCAB); MODEL,MAXLEN="xlm-roberta-base",64
tok=AutoTokenizer.from_pretrained(MODEL)
def to_ds(s):
    sub=data[data.split==s]
    return Dataset.from_dict({'text':sub['text'].tolist(),
                              'labels':[np.asarray(x,dtype=np.float32) for x in sub['labels']]})
ds={s:to_ds(s) for s in ['train','val','test']}
ds={s:d.map(lambda b:tok(b['text'],truncation=True,max_length=MAXLEN),
            batched=True,remove_columns=['text']) for s,d in ds.items()}
class ASL(torch.nn.Module):
    def __init__(s,gn=4,gp=1,clip=0.05,eps=1e-8): super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logits,y):
        xp=torch.sigmoid(logits); xn=1-xp
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        loss=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y); loss*=(1-pt).pow(g)
        return -loss.sum(1).mean()
asl=ASL()
class MLTrainer(Trainer):
    def compute_loss(s,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs); loss=asl(out.logits,labels)
        return (loss,out) if return_outputs else loss
def compute_metrics(ep):
    logits,labels=ep; probs=1/(1+np.exp(-logits)); preds=(probs>=0.5).astype(int)
    rows=np.arange(len(labels))[:,None]; top3=np.argsort(-probs,1)[:,:3]
    return {'micro_f1':f1_score(labels,preds,average='micro',zero_division=0),
            'macro_f1':f1_score(labels,preds,average='macro',zero_division=0),
            'subset_acc':accuracy_score(labels,preds),
            'P@1':float(labels[rows,probs.argmax(1)[:,None]].mean()),
            'acc@3':float(labels[rows,top3].any(1).mean())}
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=NUM,
        problem_type="multi_label_classification")
args=TrainingArguments(output_dir=f'/kaggle/working/xlmr_k{K}',num_train_epochs=2,learning_rate=2e-5,fp16=True,
    per_device_train_batch_size=128,per_device_eval_batch_size=256,eval_strategy='epoch',save_strategy='epoch',
    load_best_model_at_end=True,metric_for_best_model='acc@3',logging_steps=200,report_to='none',dataloader_num_workers=2)
trainer=MLTrainer(model=model,args=args,train_dataset=ds['train'],eval_dataset=ds['val'],
    processing_class=tok,compute_metrics=compute_metrics,data_collator=DataCollatorWithPadding(tok))
trainer.train()
print(f"\n=== K={K} TEST ==="); print(trainer.evaluate(ds['test']))
trainer.save_model(f'/kaggle/working/xlmr_k{K}_best')
print(f"ALL DONE — K={K} saved")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd, numpy as np, ast, re, unicodedata, pickle, torch
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, accuracy_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

BASE = '/kaggle/input/datasets/codewithtuqi/urdu-twitter-datasets'
K = 200         # <<< CHANGE ONLY THIS NUMBER (50, then 200). Nothing else.

# STEP 1: Excel -> parquet
proc = pd.read_excel(f'{BASE}/cleaned_urdu_dataset_1M_processed.xlsx', engine='openpyxl')
proc.to_parquet('/kaggle/working/processed.parquet', index=False)
print("STEP 1 done — rows:", len(proc))

# STEP 2: build data for this K
VS={'\uFE0F','\uFE0E'}; SKIN={chr(c) for c in range(0x1F3FB,0x1F400)}
def norm_emoji(e): return ''.join(c for c in e if c not in VS and c not in SKIN)
def parse_list(x):
    if isinstance(x,list): return x
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    try:
        v=ast.literal_eval(x); return v if isinstance(v,list) else [v]
    except Exception: return []
uniq=proc['Unique_Emojis'].apply(parse_list).apply(lambda l:list(dict.fromkeys(ne for e in l if (ne:=norm_emoji(e)))))
cnt=Counter(e for row in uniq for e in row)
VOCAB=[e for e,_ in cnt.most_common(K)]; vset=set(VOCAB)
pickle.dump(VOCAB, open(f'/kaggle/working/vocab_k{K}.pkl','wb'))
URL=re.compile(r'https?://\S+|www\.\S+')
def clean(t):
    if not isinstance(t,str): return ''
    t=unicodedata.normalize('NFC',t); t=URL.sub(' ',t)
    t=''.join(ch for ch in t if not (0xE000<=ord(ch)<=0xF8FF)); return re.sub(r'\s+',' ',t).strip()
df=pd.DataFrame({'text':proc['Urdu_Text'].apply(clean),'emojis':uniq.apply(lambda l:[e for e in l if e in vset])})
df=df[(df['emojis'].apply(len)>0)&(df['text'].str.len()>=2)].reset_index(drop=True)
Y=MultiLabelBinarizer(classes=VOCAB).fit_transform(df['emojis']).astype('int8')
idx=np.arange(len(df)); tr,tmp=train_test_split(idx,test_size=0.2,random_state=42)
va,te=train_test_split(tmp,test_size=0.5,random_state=42)
split=np.array(['train']*len(df),dtype=object); split[va]='val'; split[te]='test'
data=pd.DataFrame({'text':df['text'],'labels':list(Y),'split':split})
print(f"STEP 2 done — K={K} rows:", len(df))

# STEP 3: train
NUM=len(VOCAB); MODEL,MAXLEN="xlm-roberta-base",64
tok=AutoTokenizer.from_pretrained(MODEL)
def to_ds(s):
    sub=data[data.split==s]
    return Dataset.from_dict({'text':sub['text'].tolist(),
                              'labels':[np.asarray(x,dtype=np.float32) for x in sub['labels']]})
ds={s:to_ds(s) for s in ['train','val','test']}
ds={s:d.map(lambda b:tok(b['text'],truncation=True,max_length=MAXLEN),
            batched=True,remove_columns=['text']) for s,d in ds.items()}
class ASL(torch.nn.Module):
    def __init__(s,gn=4,gp=1,clip=0.05,eps=1e-8): super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logits,y):
        xp=torch.sigmoid(logits); xn=1-xp
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        loss=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y); loss*=(1-pt).pow(g)
        return -loss.sum(1).mean()
asl=ASL()
class MLTrainer(Trainer):
    def compute_loss(s,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs); loss=asl(out.logits,labels)
        return (loss,out) if return_outputs else loss
def compute_metrics(ep):
    logits,labels=ep; probs=1/(1+np.exp(-logits)); preds=(probs>=0.5).astype(int)
    rows=np.arange(len(labels))[:,None]; top3=np.argsort(-probs,1)[:,:3]
    return {'micro_f1':f1_score(labels,preds,average='micro',zero_division=0),
            'macro_f1':f1_score(labels,preds,average='macro',zero_division=0),
            'subset_acc':accuracy_score(labels,preds),
            'P@1':float(labels[rows,probs.argmax(1)[:,None]].mean()),
            'acc@3':float(labels[rows,top3].any(1).mean())}
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=NUM,
        problem_type="multi_label_classification")
args=TrainingArguments(output_dir=f'/kaggle/working/xlmr_k{K}',num_train_epochs=2,learning_rate=2e-5,fp16=True,
    per_device_train_batch_size=128,per_device_eval_batch_size=256,eval_strategy='epoch',save_strategy='epoch',
    load_best_model_at_end=True,metric_for_best_model='acc@3',logging_steps=200,report_to='none',dataloader_num_workers=2)
trainer=MLTrainer(model=model,args=args,train_dataset=ds['train'],eval_dataset=ds['val'],
    processing_class=tok,compute_metrics=compute_metrics,data_collator=DataCollatorWithPadding(tok))
trainer.train()
print(f"\n=== K={K} TEST ==="); print(trainer.evaluate(ds['test']))
trainer.save_model(f'/kaggle/working/xlmr_k{K}_best')
print(f"ALL DONE — K={K} saved")

In [ ]:
import pandas as pd, numpy as np, ast, matplotlib
import matplotlib.pyplot as plt
from collections import Counter
import os
os.makedirs('/kaggle/working/figs', exist_ok=True)

BASE='/kaggle/input/datasets/codewithtuqi/urdu-twitter-datasets'
proc = pd.read_excel(f'{BASE}/cleaned_urdu_dataset_1M_processed.xlsx', engine='openpyxl')

VS={'\uFE0F','\uFE0E'}; SKIN={chr(c) for c in range(0x1F3FB,0x1F400)}
def norm(e): return ''.join(c for c in e if c not in VS and c not in SKIN)
def pl(x):
    try: v=ast.literal_eval(x); return v if isinstance(v,list) else [v]
    except Exception: return []
uniq = proc['Unique_Emojis'].apply(pl).apply(lambda l:[norm(e) for e in l if norm(e)])
cnt = Counter(e for r in uniq for e in r)

# FIG A1: rank-frequency (log-log) with K cutoffs
freqs = np.array(sorted(cnt.values(), reverse=True))
fig,ax = plt.subplots(figsize=(4.6,3.2))
ax.loglog(range(1,len(freqs)+1), freqs, lw=1.5, color='#1a5fb4')
for k in [20,50,100,200]:
    ax.axvline(k, ls=':', color='#c01c28', lw=1)
    ax.annotate(f'K={k}', (k, freqs[min(k,len(freqs)-1)]), fontsize=7,
                rotation=90, va='bottom', ha='right', color='#c01c28')
ax.set_xlabel('Emoji rank (log)'); ax.set_ylabel('Frequency (log)')
ax.grid(alpha=0.3, which='both', lw=0.4); plt.tight_layout()
plt.savefig('/kaggle/working/figs/freq_dist.pdf'); plt.savefig('/kaggle/working/figs/freq_dist.png', dpi=160); plt.close()

# FIG A2: label cardinality histogram
card = uniq.apply(len).clip(upper=8)
fig,ax = plt.subplots(figsize=(4.2,3.0))
vc = card.value_counts().sort_index()
ax.bar(vc.index, vc.values/len(card)*100, color='#1a5fb4', width=0.7)
ax.set_xlabel('Distinct emojis per tweet (8 = 8+)'); ax.set_ylabel('% of tweets')
ax.grid(alpha=0.3, axis='y', lw=0.4); plt.tight_layout()
plt.savefig('/kaggle/working/figs/cardinality.pdf'); plt.savefig('/kaggle/working/figs/cardinality.png', dpi=160); plt.close()

# FIG A3: cumulative coverage curve
cum = np.cumsum(freqs)/freqs.sum()
fig,ax = plt.subplots(figsize=(4.2,3.0))
ax.plot(range(1,len(cum)+1), cum*100, lw=1.6, color='#1a5fb4')
ax.set_xscale('log'); ax.set_xlabel('Inventory size $K$ (log)')
ax.set_ylabel('% of emoji occurrences covered')
for k,c in [(20,None),(50,None),(100,None),(200,None)]:
    ax.axvline(k, ls=':', lw=0.8, color='#c01c28')
ax.grid(alpha=0.3, lw=0.4); plt.tight_layout()
plt.savefig('/kaggle/working/figs/coverage.pdf'); plt.savefig('/kaggle/working/figs/coverage.png', dpi=160); plt.close()
print("Stage A figures saved to /kaggle/working/figs/")

In [ ]:
# ===== STEP 4+5: predictions, multi-label examples, and all figures =====
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, os, json
from sklearn.metrics import f1_score, precision_recall_curve, average_precision_score
try:
    import emoji as emojilib
except Exception:
    os.system('pip -q install emoji'); import emoji as emojilib
os.makedirs('/kaggle/working/figs', exist_ok=True)
def ename(ch):
    d = emojilib.demojize(ch); return d.strip(':').replace('_',' ') if d.startswith(':') else ch

# --- predictions on val/test ---
pv = trainer.predict(ds['val']);  pt = trainer.predict(ds['test'])
probs_v = 1/(1+np.exp(-pv.predictions)); yv = pv.label_ids.astype(int)
probs_t = 1/(1+np.exp(-pt.predictions)); yt = pt.label_ids.astype(int)

# --- per-class calibrated thresholds (tuned on val) ---
grid = np.arange(0.05, 0.60, 0.02)
thr = np.array([max(grid, key=lambda t: f1_score(yv[:,c], (probs_v[:,c]>=t).astype(int), zero_division=0))
                for c in range(yv.shape[1])])
pred_set = (probs_t >= thr)
np.savez('/kaggle/working/test_preds_k%d.npz' % K, probs=probs_t, y=yt, thr=thr)

# --- multi-label examples for Table 6 ---
test_texts = data[data.split=='test']['text'].tolist()
multi = np.where(yt.sum(1) >= 2)[0]
def row_f1(i):
    inter = int((pred_set[i] & (yt[i]==1)).sum()); p,g = int(pred_set[i].sum()), int(yt[i].sum())
    return 2*inter/(p+g) if (p+g) else 0.0
ranked = sorted(multi.tolist(), key=row_f1, reverse=True)
show = ranked[:10] + ranked[len(ranked)//2: len(ranked)//2+5]
print("\n===== MULTI-LABEL EXAMPLES (copy this to me) =====")
for i in show:
    gold = [VOCAB[k] for k in np.where(yt[i]==1)[0]]
    pred = [VOCAB[k] for k in np.where(pred_set[i])[0]]
    top5 = [(VOCAB[k], round(float(probs_t[i,k]),2)) for k in np.argsort(-probs_t[i])[:5]]
    print("TEXT:", test_texts[i][:100]); print("GOLD:", gold, "| PRED-SET:", pred, "| TOP5:", top5); print("-"*70)

freq_order = np.argsort(-yt.sum(0))

# FIG B1: per-class F1, 15 most frequent
f1c = [f1_score(yt[:,c], pred_set[:,c].astype(int), zero_division=0) for c in freq_order[:15]]
names = [ename(VOCAB[c])[:18] for c in freq_order[:15]]
fig,ax = plt.subplots(figsize=(5.4,3.4))
ax.barh(range(len(f1c))[::-1], f1c, color='#1a5fb4')
ax.set_yticks(range(len(f1c))[::-1]); ax.set_yticklabels(names, fontsize=7)
ax.set_xlabel('Per-class F1 (calibrated)'); ax.grid(alpha=0.3, axis='x', lw=0.4)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/perclass_f1.pdf'); plt.savefig('/kaggle/working/figs/perclass_f1.png',dpi=160); plt.close()

# FIG B2: precision-recall
fig,ax = plt.subplots(figsize=(4.6,3.4))
P,R,_ = precision_recall_curve(yt.ravel(), probs_t.ravel())
ax.plot(R,P, lw=1.8, color='black', label=f'micro (AP={average_precision_score(yt,probs_t,average="micro"):.3f})')
for c,col in [(freq_order[0],'#1a5fb4'),(freq_order[len(freq_order)//2],'#26a269'),(freq_order[-1],'#c01c28')]:
    P,R,_ = precision_recall_curve(yt[:,c], probs_t[:,c]); ax.plot(R,P, lw=1.2, color=col, label=ename(VOCAB[c])[:20])
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.legend(fontsize=7, frameon=False); ax.grid(alpha=0.3,lw=0.4)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/pr_curves.pdf'); plt.savefig('/kaggle/working/figs/pr_curves.png',dpi=160); plt.close()

# FIG B3: threshold sweep
ths = np.arange(0.05,0.91,0.05)
mi = [f1_score(yt,(probs_t>=t).astype(int),average='micro',zero_division=0) for t in ths]
ma = [f1_score(yt,(probs_t>=t).astype(int),average='macro',zero_division=0) for t in ths]
fig,ax = plt.subplots(figsize=(4.4,3.2))
ax.plot(ths,mi,'o-',ms=3.5,lw=1.6,color='#1a5fb4',label='micro-F1'); ax.plot(ths,ma,'s--',ms=3.5,lw=1.6,color='#c01c28',label='macro-F1')
ax.set_xlabel('Global decision threshold'); ax.set_ylabel('F1'); ax.legend(frameon=False,fontsize=8); ax.grid(alpha=0.3,lw=0.4)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/threshold_sweep.pdf'); plt.savefig('/kaggle/working/figs/threshold_sweep.png',dpi=160); plt.close()

# FIG B4: confusion heatmap (single-label test subset, top-15)
single = np.where(yt.sum(1)==1)[0]; top15=list(freq_order[:15]); idxmap={c:i for i,c in enumerate(top15)}
M = np.zeros((15,16))
for i in single:
    g=int(np.where(yt[i]==1)[0][0])
    if g not in idxmap: continue
    M[idxmap[g], idxmap.get(int(probs_t[i].argmax()),15)] += 1
M = M/np.clip(M.sum(1,keepdims=True),1,None)
fig,ax = plt.subplots(figsize=(5.6,4.6)); im=ax.imshow(M,cmap='Blues',vmin=0,vmax=M.max())
ax.set_xticks(range(16)); ax.set_xticklabels([ename(VOCAB[c])[:10] for c in top15]+['other'],rotation=90,fontsize=6)
ax.set_yticks(range(15)); ax.set_yticklabels([ename(VOCAB[c])[:14] for c in top15],fontsize=6)
ax.set_xlabel('Predicted (top-1)'); ax.set_ylabel('Gold (single-label)'); fig.colorbar(im,fraction=0.04)
plt.tight_layout(); plt.savefig('/kaggle/working/figs/confusion_top15.pdf'); plt.savefig('/kaggle/working/figs/confusion_top15.png',dpi=160); plt.close()

# FIG B5: t-SNE of learned emoji (classifier head) vectors
from sklearn.manifold import TSNE
W = trainer.model.classifier.out_proj.weight.detach().cpu().numpy()
Z = TSNE(n_components=2, perplexity=15, random_state=0, init='pca').fit_transform(W)
def cat(e):
    cp=ord(e[0])
    if 0x1F600<=cp<=0x1F64F: return ('face','#1a5fb4')
    if e[0] in '❤♥💔💕💞💝💖💗💘💙💚💛💜🖤💟' or 0x1F493<=cp<=0x1F49F: return ('heart','#c01c28')
    if 0x1F330<=cp<=0x1F344 or e[0] in '🌹🌷🌸🌺🌻🌼💐🌿🍀🌴🌾': return ('nature','#26a269')
    return ('other','#5e5c64')
fig,ax = plt.subplots(figsize=(5.2,4.4)); seen=set()
for k,e in enumerate(VOCAB):
    lab,col=cat(e); ax.scatter(Z[k,0],Z[k,1],s=14,color=col,label=lab if lab not in seen else None); seen.add(lab)
for k in freq_order[:12]: ax.annotate(ename(VOCAB[k])[:12],(Z[k,0],Z[k,1]),fontsize=6,alpha=0.85)
ax.legend(fontsize=7,frameon=False); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig('/kaggle/working/figs/tsne_emoji.pdf'); plt.savefig('/kaggle/working/figs/tsne_emoji.png',dpi=160); plt.close()

json.dump(trainer.state.log_history, open('/kaggle/working/figs/log_history.json','w'))
print("\nDONE — figures in /kaggle/working/figs/")